# 第 32 章：一维卷积与光谱局部模式

这个 notebook 把上一章的卷积直觉从二维 cutout 平移到一维光谱窗口：

- 先用 raw flux bins 建立最近中心 baseline
- 再手写三个最小一维卷积核
- 比较两者在轻微谱线位移下的表现差异
- 再引入一个带 continuum slope、quality flag 和 clean/review route 的第二教学 workflow
- 回到局部峰、谷和双峰结构，理解一维卷积为什么适合光谱

教学说明：这里默认仍然不依赖 NumPy、PyTorch 或专门的光谱库；notebook 会先用纯 Python 训练一个带 validation 的极小 `Conv1d` workflow learner，再在环境允许时给出对应的 `torch` 框架对照。


In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

DATA_PATH = Path("../../data/small/spectral_conv1d_demo.csv").resolve()
WINDOW_LENGTH = 14

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "spectral_label": raw["spectral_label"],
            "split": raw["split"],
        }
        for index in range(WINDOW_LENGTH):
            parsed[f"f{index:02d}"] = float(raw[f"f{index:02d}"])
        rows.append(parsed)

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]


def spectrum_from_row(row):
    return [row[f"f{index:02d}"] for index in range(WINDOW_LENGTH)]


print(f"Loaded {len(rows)} spectral windows from {DATA_PATH.name}")
print("class counts:", dict(Counter(row["spectral_label"] for row in rows)))
print("train/test sizes:", len(train_rows), len(test_rows))
print("test ids:", [row["sample_id"] for row in test_rows])
print("Representative spectra:")
for sample_id in ["S01", "A01", "D01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    rounded_flux = [round(value, 2) for value in spectrum_from_row(row)]
    print(sample_id, row["spectral_label"], rounded_flux)


In [ ]:
import math


def centroid(vectors):
    return [sum(values) / len(values) for values in zip(*vectors)]


def euclidean_distance(left, right):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(left, right)))


def nearest_centroid(train_items, feature_fn, test_items):
    labels = sorted({item["spectral_label"] for item in train_items})
    centers = {}
    for label in labels:
        vectors = [feature_fn(item) for item in train_items if item["spectral_label"] == label]
        centers[label] = centroid(vectors)

    predictions = []
    for item in test_items:
        vector = feature_fn(item)
        ranked = sorted(
            ((label, euclidean_distance(vector, centers[label])) for label in labels),
            key=lambda pair: pair[1],
        )
        predictions.append({
            "sample_id": item["sample_id"],
            "true_label": item["spectral_label"],
            "predicted_label": ranked[0][0],
            "distances": {label: round(distance, 3) for label, distance in ranked},
        })
    return centers, predictions


def accuracy(predictions):
    return sum(item["true_label"] == item["predicted_label"] for item in predictions) / len(predictions)


def confusion(predictions):
    matrix = {}
    for item in predictions:
        truth = item["true_label"]
        predicted = item["predicted_label"]
        matrix.setdefault(truth, {})
        matrix[truth][predicted] = matrix[truth].get(predicted, 0) + 1
    return matrix


raw_centers, raw_predictions = nearest_centroid(train_rows, spectrum_from_row, test_rows)

print("Raw-feature nearest-centroid baseline:")
for item in raw_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("raw accuracy:", round(accuracy(raw_predictions), 3))
print("raw confusion matrix:", confusion(raw_predictions))


In [ ]:
kernels = {
    "peak_detector": [-1.0, 2.0, -1.0],
    "trough_detector": [1.0, -2.0, 1.0],
    "double_peak_detector": [1.0, -1.0, -1.0, 1.0],
}


def conv1d_valid(signal, kernel):
    kernel_size = len(kernel)
    return [
        sum(signal[index + offset] * kernel[offset] for offset in range(kernel_size))
        for index in range(len(signal) - kernel_size + 1)
    ]


def relu(values):
    return [max(0.0, value) for value in values]


def max_pool_1d(values, size=2, stride=2):
    return [max(values[index:index + size]) for index in range(0, len(values) - size + 1, stride)]


def summarize_kernel_response(signal):
    summary = []
    detailed = {}
    for kernel_name, kernel in kernels.items():
        pooled = max_pool_1d(relu(conv1d_valid(signal, kernel)))
        summary.extend([max(pooled), sum(pooled) / len(pooled)])
        detailed[kernel_name] = pooled
    return summary, detailed


print("Pooled responses for representative training samples:")
for sample_id in ["S01", "A01", "D01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    feature_summary, detailed = summarize_kernel_response(spectrum_from_row(row))
    print("\n", sample_id, row["spectral_label"])
    print("feature summary:", [round(value, 3) for value in feature_summary])
    for kernel_name, pooled in detailed.items():
        print(kernel_name, [round(value, 3) for value in pooled])


In [ ]:
def conv_features(item):
    summary, _ = summarize_kernel_response(spectrum_from_row(item))
    return summary


conv_centers, conv_predictions = nearest_centroid(train_rows, conv_features, test_rows)

print("One-dimensional convolution feature model:")
for item in conv_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("conv accuracy:", round(accuracy(conv_predictions), 3))
print("conv confusion matrix:", confusion(conv_predictions))
print("accuracy improvement over raw baseline:", round(accuracy(conv_predictions) - accuracy(raw_predictions), 3))


In [ ]:
print("Shift diagnostics:")
for sample_id in ["S01", "S03", "D01", "D03"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    summary, detailed = summarize_kernel_response(spectrum_from_row(row))
    print(sample_id, row["split"], row["spectral_label"], [round(value, 3) for value in summary])
    print("peak_detector =", [round(value, 3) for value in detailed["peak_detector"]])
    print("double_peak_detector =", [round(value, 3) for value in detailed["double_peak_detector"]])

print("Interpretation:")
print("  The raw baseline ties the class to exact bin positions, so a shifted narrow peak can look closer to the double-peak centroid.")
print("  The one-dimensional convolution kernels keep scanning the whole window, so they still fire when the same local line shape appears elsewhere.")
print("  Max pooling then keeps the strongest local response instead of forcing the model to memorize the absolute coordinate.")


In [ ]:
WORKFLOW_DATA_PATH = Path("../../data/small/spectral_conv1d_workflow_demo.csv").resolve()
WORKFLOW_LENGTH = 18

with WORKFLOW_DATA_PATH.open(newline="", encoding="utf-8") as handle:
    workflow_rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "spectral_label": raw["spectral_label"],
            "split_role": raw["split_role"],
            "quality_flag": raw["quality_flag"],
        }
        for index in range(WORKFLOW_LENGTH):
            parsed[f"f{index:02d}"] = float(raw[f"f{index:02d}"])
        workflow_rows.append(parsed)


def workflow_spectrum_from_row(row):
    return [row[f"f{index:02d}"] for index in range(WORKFLOW_LENGTH)]


def estimate_linear_continuum(signal, edge_bins=3):
    left_level = sum(signal[:edge_bins]) / edge_bins
    right_level = sum(signal[-edge_bins:]) / edge_bins
    return [
        left_level + (index / (len(signal) - 1)) * (right_level - left_level)
        for index in range(len(signal))
    ]


def continuum_normalize(signal):
    baseline = estimate_linear_continuum(signal)
    return [(value - base) / (base or 1.0) for value, base in zip(signal, baseline)]


def workflow_route(row):
    return "manual_review" if row["quality_flag"] != "clean" else "ready_for_model"


workflow_clean_rows = [row for row in workflow_rows if row["quality_flag"] == "clean"]
workflow_train_rows = [row for row in workflow_clean_rows if row["split_role"] == "train"]
workflow_validation_rows = [row for row in workflow_clean_rows if row["split_role"] == "validation"]
workflow_test_rows = [row for row in workflow_clean_rows if row["split_role"] == "test"]
workflow_review_rows = [row for row in workflow_rows if row["quality_flag"] != "clean"]

print(f"Loaded {len(workflow_rows)} workflow spectra from {WORKFLOW_DATA_PATH.name}")
print("split-role counts:", dict(Counter(row["split_role"] for row in workflow_rows)))
print("quality counts:", dict(Counter(row["quality_flag"] for row in workflow_rows)))
print("route counts:", dict(Counter(workflow_route(row) for row in workflow_rows)))
print("clean train/validation/test sizes:", len(workflow_train_rows), len(workflow_validation_rows), len(workflow_test_rows))
print("review ids:", [row["sample_id"] for row in workflow_review_rows])
print("Workflow examples:")
for sample_id in ["T_na0", "X_na0", "R_art"]:
    row = next(item for item in workflow_rows if item["sample_id"] == sample_id)
    raw_signal = [round(value, 3) for value in workflow_spectrum_from_row(row)]
    normalized_signal = [round(value, 3) for value in continuum_normalize(workflow_spectrum_from_row(row))]
    print(f"\n{sample_id} {row['split_role']} {row['quality_flag']} {row['spectral_label']}")
    print("raw =", raw_signal)
    print("continuum-normalized =", normalized_signal)


In [ ]:
workflow_kernels = {
    "peak_detector": [-0.6, 1.2, -0.6],
    "trough_detector": [0.7, -1.4, 0.7],
    "double_peak_detector": [0.8, -0.6, -0.4, -0.6, 0.8],
}


def workflow_summarize_kernel_response(signal):
    summary = []
    detailed = {}
    for kernel_name, kernel in workflow_kernels.items():
        pooled = max_pool_1d(relu(conv1d_valid(signal, kernel)))
        summary.extend([max(pooled), sum(pooled) / len(pooled)])
        detailed[kernel_name] = pooled
    return summary, detailed


def workflow_report(name, feature_fn):
    _, predictions = nearest_centroid(workflow_train_rows, feature_fn, workflow_test_rows)
    print(f"\n{name}")
    for item in predictions:
        print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
    print("accuracy:", round(accuracy(predictions), 3))
    print("confusion matrix:", confusion(predictions))
    return predictions


workflow_raw_predictions = workflow_report(
    "Workflow baseline: raw spectral windows on clean test",
    lambda item: workflow_spectrum_from_row(item),
)
workflow_normalized_raw_predictions = workflow_report(
    "Workflow baseline: continuum-normalized raw windows on clean test",
    lambda item: continuum_normalize(workflow_spectrum_from_row(item)),
)
workflow_conv_predictions = workflow_report(
    "Workflow model: normalized convolution features on clean test",
    lambda item: workflow_summarize_kernel_response(continuum_normalize(workflow_spectrum_from_row(item)))[0],
)

print("\nManual review queue:")
for row in workflow_review_rows:
    print(row["sample_id"], row["spectral_label"], row["quality_flag"], workflow_route(row))

print("Workflow summary metrics:")
print("  raw window accuracy =", round(accuracy(workflow_raw_predictions), 3))
print("  normalized raw window accuracy =", round(accuracy(workflow_normalized_raw_predictions), 3))
print("  normalized conv-feature accuracy =", round(accuracy(workflow_conv_predictions), 3))
print("  review-queue capture =", f"{len(workflow_review_rows)}/{len(workflow_review_rows)}")

print("Interpretation:")
print("  Continuum normalization removes part of the slope mismatch, so the raw-window baseline improves from 0.50 to 0.667.")
print("  But the normalized raw window still confuses shifted narrow peaks with double-peak structure.")
print("  The widened local convolution templates recover the reusable line shape and reach 1.0 on the clean test split.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")
else:
    figure, axes = plt.subplots(2, 3, figsize=(9, 5), sharex=True, sharey=True)
    x_values = list(range(WINDOW_LENGTH))
    for axis, sample_id in zip(axes.flat, ["S01", "A01", "D01", "S03", "A03", "D03"]):
        row = next(item for item in rows if item["sample_id"] == sample_id)
        axis.plot(x_values, spectrum_from_row(row), marker="o")
        axis.set_title(f"{sample_id}\n{row['spectral_label']}")
        axis.set_xlabel("flux bin")
        axis.set_ylabel("normalized flux")
        axis.grid(alpha=0.3)
    figure.suptitle("Tiny spectral windows for one-dimensional convolution teaching")
    figure.tight_layout()
    plt.show()


In [ ]:
import copy
import random

WORKFLOW_SEED = 1
TRAINABLE_KERNEL_SIZE = 3
TRAINABLE_FILTERS = 3
TRAINABLE_EPOCHS = 1200
TRAINABLE_LEARNING_RATE = 0.03

workflow_label_order = sorted({row["spectral_label"] for row in workflow_clean_rows})
workflow_label_to_index = {label: index for index, label in enumerate(workflow_label_order)}


def softmax(values):
    shifted = [value - max(values) for value in values]
    exp_values = [math.exp(value) for value in shifted]
    total = sum(exp_values)
    return [value / total for value in exp_values]


def argmax(values):
    return max(range(len(values)), key=lambda index: values[index])


class TinyConv1dClassifier:
    def __init__(self, seed=WORKFLOW_SEED, kernel_size=TRAINABLE_KERNEL_SIZE, num_filters=TRAINABLE_FILTERS):
        rng = random.Random(seed)
        self.kernel_size = kernel_size
        self.num_filters = num_filters
        self.output_length = WORKFLOW_LENGTH - kernel_size + 1
        self.filters = [
            [rng.uniform(-0.25, 0.25) for _ in range(kernel_size)]
            for _ in range(num_filters)
        ]
        self.filter_bias = [0.0 for _ in range(num_filters)]
        self.classifier = [
            [rng.uniform(-0.25, 0.25) for _ in range(num_filters)]
            for _ in range(len(workflow_label_order))
        ]
        self.classifier_bias = [0.0 for _ in range(len(workflow_label_order))]

    def forward(self, signal):
        conv_pre = []
        max_positions = []
        pooled = []
        for filter_index in range(self.num_filters):
            kernel = self.filters[filter_index]
            bias = self.filter_bias[filter_index]
            raw = []
            for start in range(self.output_length):
                value = bias
                for offset in range(self.kernel_size):
                    value += kernel[offset] * signal[start + offset]
                raw.append(value)
            activated = [max(0.0, value) for value in raw]
            peak_index = max(range(len(activated)), key=lambda index: activated[index])
            conv_pre.append(raw)
            max_positions.append(peak_index)
            pooled.append(activated[peak_index])

        logits = []
        for class_index in range(len(workflow_label_order)):
            logits.append(
                self.classifier_bias[class_index]
                + sum(
                    self.classifier[class_index][filter_index] * pooled[filter_index]
                    for filter_index in range(self.num_filters)
                )
            )
        probabilities = softmax(logits)
        return {
            "conv_pre": conv_pre,
            "max_positions": max_positions,
            "pooled": pooled,
            "probabilities": probabilities,
            "predicted_label": workflow_label_order[argmax(probabilities)],
            "confidence": max(probabilities),
        }

    def loss_and_gradients(self, row):
        signal = continuum_normalize(workflow_spectrum_from_row(row))
        summary = self.forward(signal)
        target_index = workflow_label_to_index[row["spectral_label"]]
        probabilities = summary["probabilities"]
        loss = -math.log(max(probabilities[target_index], 1e-12))

        grad_filters = [[0.0 for _ in range(self.kernel_size)] for _ in range(self.num_filters)]
        grad_filter_bias = [0.0 for _ in range(self.num_filters)]
        grad_classifier = [
            [0.0 for _ in range(self.num_filters)]
            for _ in range(len(workflow_label_order))
        ]
        grad_classifier_bias = [0.0 for _ in range(len(workflow_label_order))]

        d_logits = probabilities[:]
        d_logits[target_index] -= 1.0
        for class_index in range(len(workflow_label_order)):
            grad_classifier_bias[class_index] += d_logits[class_index]
            for filter_index in range(self.num_filters):
                grad_classifier[class_index][filter_index] += (
                    d_logits[class_index] * summary["pooled"][filter_index]
                )

        d_pooled = [
            sum(
                self.classifier[class_index][filter_index] * d_logits[class_index]
                for class_index in range(len(workflow_label_order))
            )
            for filter_index in range(self.num_filters)
        ]

        for filter_index in range(self.num_filters):
            peak_index = summary["max_positions"][filter_index]
            if summary["conv_pre"][filter_index][peak_index] <= 0.0:
                continue
            gradient_at_peak = d_pooled[filter_index]
            grad_filter_bias[filter_index] += gradient_at_peak
            for offset in range(self.kernel_size):
                grad_filters[filter_index][offset] += gradient_at_peak * signal[peak_index + offset]

        return loss, {
            "filters": grad_filters,
            "filter_bias": grad_filter_bias,
            "classifier": grad_classifier,
            "classifier_bias": grad_classifier_bias,
        }

    def train(self, train_items, validation_items, epochs=TRAINABLE_EPOCHS, learning_rate=TRAINABLE_LEARNING_RATE):
        snapshot_epochs = {1, 120, 480, epochs}
        snapshots = {}
        best_epoch = 0
        best_validation_accuracy = -1.0
        best_validation_loss = float("inf")
        best_state = copy.deepcopy((self.filters, self.filter_bias, self.classifier, self.classifier_bias))

        for epoch in range(1, epochs + 1):
            total_loss = 0.0
            grad_filters = [[0.0 for _ in range(self.kernel_size)] for _ in range(self.num_filters)]
            grad_filter_bias = [0.0 for _ in range(self.num_filters)]
            grad_classifier = [
                [0.0 for _ in range(self.num_filters)]
                for _ in range(len(workflow_label_order))
            ]
            grad_classifier_bias = [0.0 for _ in range(len(workflow_label_order))]

            for row in train_items:
                loss, gradients = self.loss_and_gradients(row)
                total_loss += loss
                for filter_index in range(self.num_filters):
                    grad_filter_bias[filter_index] += gradients["filter_bias"][filter_index]
                    for offset in range(self.kernel_size):
                        grad_filters[filter_index][offset] += gradients["filters"][filter_index][offset]
                for class_index in range(len(workflow_label_order)):
                    grad_classifier_bias[class_index] += gradients["classifier_bias"][class_index]
                    for filter_index in range(self.num_filters):
                        grad_classifier[class_index][filter_index] += gradients["classifier"][class_index][filter_index]

            sample_count = len(train_items)
            for filter_index in range(self.num_filters):
                self.filter_bias[filter_index] -= learning_rate * grad_filter_bias[filter_index] / sample_count
                for offset in range(self.kernel_size):
                    self.filters[filter_index][offset] -= learning_rate * grad_filters[filter_index][offset] / sample_count
            for class_index in range(len(workflow_label_order)):
                self.classifier_bias[class_index] -= learning_rate * grad_classifier_bias[class_index] / sample_count
                for filter_index in range(self.num_filters):
                    self.classifier[class_index][filter_index] -= learning_rate * grad_classifier[class_index][filter_index] / sample_count

            validation_predictions = []
            validation_loss = 0.0
            for row in validation_items:
                summary = self.forward(continuum_normalize(workflow_spectrum_from_row(row)))
                validation_predictions.append(summary["predicted_label"])
                target_index = workflow_label_to_index[row["spectral_label"]]
                validation_loss += -math.log(max(summary["probabilities"][target_index], 1e-12))
            validation_accuracy = sum(
                predicted == row["spectral_label"]
                for predicted, row in zip(validation_predictions, validation_items)
            ) / len(validation_items)
            validation_loss /= len(validation_items)

            if (
                validation_accuracy > best_validation_accuracy
                or (
                    validation_accuracy == best_validation_accuracy
                    and validation_loss < best_validation_loss
                )
            ):
                best_epoch = epoch
                best_validation_accuracy = validation_accuracy
                best_validation_loss = validation_loss
                best_state = copy.deepcopy((self.filters, self.filter_bias, self.classifier, self.classifier_bias))

            if epoch in snapshot_epochs:
                snapshots[epoch] = {
                    "train_loss": total_loss / sample_count,
                    "validation_accuracy": validation_accuracy,
                    "validation_loss": validation_loss,
                }

        self.filters, self.filter_bias, self.classifier, self.classifier_bias = copy.deepcopy(best_state)
        return snapshots, best_epoch, best_validation_accuracy, best_validation_loss


tiny_conv_model = TinyConv1dClassifier()
training_snapshots, best_epoch, best_validation_accuracy, best_validation_loss = tiny_conv_model.train(
    workflow_train_rows,
    workflow_validation_rows,
)

validation_summaries = []
for row in workflow_validation_rows:
    summary = tiny_conv_model.forward(continuum_normalize(workflow_spectrum_from_row(row)))
    validation_summaries.append({
        "sample_id": row["sample_id"],
        "true_label": row["spectral_label"],
        "predicted_label": summary["predicted_label"],
        "confidence": summary["confidence"],
    })

workflow_trainable_ready_threshold = min(item["confidence"] for item in validation_summaries) - 0.02

trainable_test_predictions = []
trainable_routes = []
print("\nTrainable tiny Conv1d workflow:")
print(
    "training snapshots:",
    {
        epoch: {
            metric_name: round(metric_value, 4)
            for metric_name, metric_value in snapshot.items()
        }
        for epoch, snapshot in training_snapshots.items()
    },
)
print("best validation accuracy:", round(best_validation_accuracy, 3), "at epoch", best_epoch)
print("best validation loss:", round(best_validation_loss, 4))
print(
    "validation summaries:",
    [
        (
            item["sample_id"],
            item["true_label"],
            item["predicted_label"],
            round(item["confidence"], 4),
        )
        for item in validation_summaries
    ],
)
print("ready threshold =", round(workflow_trainable_ready_threshold, 4))

for row in workflow_test_rows + workflow_review_rows:
    summary = tiny_conv_model.forward(continuum_normalize(workflow_spectrum_from_row(row)))
    peak_windows = [
        f"f{peak_index:02d}-f{peak_index + TRAINABLE_KERNEL_SIZE - 1:02d}"
        for peak_index in summary["max_positions"]
    ]
    route = (
        "manual_review"
        if row["quality_flag"] != "clean" or summary["confidence"] < workflow_trainable_ready_threshold
        else "conv_prediction_ready"
    )
    trainable_routes.append({
        "sample_id": row["sample_id"],
        "quality_flag": row["quality_flag"],
        "true_label": row["spectral_label"],
        "predicted_label": summary["predicted_label"],
        "confidence": summary["confidence"],
        "route": route,
    })
    if row["quality_flag"] == "clean":
        trainable_test_predictions.append({
            "sample_id": row["sample_id"],
            "true_label": row["spectral_label"],
            "predicted_label": summary["predicted_label"],
        })
    print(
        row["sample_id"],
        row["spectral_label"],
        "->",
        summary["predicted_label"],
        "confidence=",
        round(summary["confidence"], 4),
        "route=",
        route,
        "peak_windows=",
        peak_windows,
        "pooled=",
        [round(value, 3) for value in summary["pooled"]],
    )

print("trainable clean-test accuracy:", round(accuracy(trainable_test_predictions), 3))
print("trainable clean-test confusion matrix:", confusion(trainable_test_predictions))
print("trainable route counts:", dict(Counter(item["route"] for item in trainable_routes)))
print(
    "trainable manual review ids:",
    [item["sample_id"] for item in trainable_routes if item["route"] == "manual_review"],
)
print("learned kernels:")
for filter_index, kernel in enumerate(tiny_conv_model.filters):
    print(filter_index, [round(value, 3) for value in kernel], "bias", round(tiny_conv_model.filter_bias[filter_index], 3))
print("classifier head weights:")
for label, weights, bias in zip(workflow_label_order, tiny_conv_model.classifier, tiny_conv_model.classifier_bias):
    print(label, [round(value, 3) for value in weights], "bias", round(bias, 3))
print("Interpretation:")
print("  The first learned kernel grows into an emission-like center-positive detector, while another kernel becomes useful for the absorption trough.")
print("  On the clean test split this tiny trainable Conv1d reaches 1.0 classification accuracy with only three learned 3-bin filters and a linear head.")
print("  The validation-calibrated confidence threshold still holds X_do1 for manual review, and R_low / R_art remain outside the auto-accept path because workflow routing matters as much as the label itself.")


In [ ]:
try:
    import copy
    import torch
except ModuleNotFoundError as exc:
    print(f"Optional PyTorch comparison skipped: {exc}")
else:
    torch.manual_seed(WORKFLOW_SEED)

    def tensorize_workflow(items):
        x_tensor = torch.tensor(
            [continuum_normalize(workflow_spectrum_from_row(row)) for row in items],
            dtype=torch.float32,
        ).unsqueeze(1)
        y_tensor = torch.tensor(
            [workflow_label_to_index[row["spectral_label"]] for row in items],
            dtype=torch.long,
        )
        return x_tensor, y_tensor

    x_train, y_train = tensorize_workflow(workflow_train_rows)
    x_validation, y_validation = tensorize_workflow(workflow_validation_rows)
    x_test, y_test = tensorize_workflow(workflow_test_rows)

    model = torch.nn.Sequential(
        torch.nn.Conv1d(1, TRAINABLE_FILTERS, kernel_size=TRAINABLE_KERNEL_SIZE),
        torch.nn.ReLU(),
        torch.nn.AdaptiveMaxPool1d(1),
        torch.nn.Flatten(),
        torch.nn.Linear(TRAINABLE_FILTERS, len(workflow_label_order)),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAINABLE_LEARNING_RATE)
    loss_fn = torch.nn.CrossEntropyLoss()
    best_state = copy.deepcopy(model.state_dict())
    best_validation_accuracy = -1.0
    best_epoch = 0

    for epoch in range(1, TRAINABLE_EPOCHS + 1):
        optimizer.zero_grad()
        logits = model(x_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            validation_accuracy = (model(x_validation).argmax(dim=1) == y_validation).float().mean().item()
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    with torch.no_grad():
        test_logits = model(x_test)
        predicted = test_logits.argmax(dim=1)
        test_accuracy = (predicted == y_test).float().mean().item()
        reverse_labels = {index: label for label, index in workflow_label_to_index.items()}
        print("Optional PyTorch Conv1d workflow model:")
        print("best validation accuracy:", round(best_validation_accuracy, 3), "at epoch", best_epoch)
        print("test accuracy:", round(test_accuracy, 3))
        for row, pred_index in zip(workflow_test_rows, predicted.tolist()):
            print("torch", row["sample_id"], row["spectral_label"], "->", reverse_labels[pred_index])
        print("learned first-layer kernels:")
        for kernel_index, kernel in enumerate(model[0].weight.squeeze(1).tolist()):
            print(kernel_index, [round(value, 3) for value in kernel])
